# Tables, Data Types and Primary Keys
## Week 3 · Member 7 · Harbour learning programme

**By the end:** Design a table, choose suitable column types and a primary key, and justify those decisions.

**Your Friday deliverable:** Create the Library Loans API's `books` and `members` tables with constraints, then explain each choice to a reviewer.

**Context:** The [Harbour learning plan](https://github.com/Snrboi/harbour/blob/learn/README.md#week-3-sql-and-postgresql-basics) uses PostgreSQL. Library Loans API and Harbour are **separate projects**. We learn with library tables, then apply the reasoning to a **proposed** Harbour document table.

**Class format:** About 75–90 minutes: concepts (20), library design and tests (35), Harbour transfer (10), independent design and review (15–25).

Use fictional data only. Predict each result before running it. This notebook is teaching material—not an agreed production schema or an Alembic migration.

## 1. Start with the thing one row represents

A **table** stores records of the same kind. A **row** is one record. A **column** is one named attribute with a declared type.

| book_id | title | author | publication_year |
|---:|---|---|---:|
| 1 | The Lantern Map | Ada Fiction | 2021 |
| 2 | The Lantern Map | Boma Fiction | 2023 |

**Row meaning:** One row represents one catalogue book record, not one physical copy and not one loan.

Two rows can have the same title. The IDs distinguish them.

Before writing SQL, complete: **“One row in this table represents ___.”** If your answer mixes a book, a borrower, and a loan, separate the responsibilities before choosing columns.

**Ask the class:** Why is `title` not enough to identify one book record?

## 2. Design in six decisions

1. **Row meaning:** What does one record represent?
2. **Columns:** What facts about that record must we store?
3. **Types:** What operations and valid values must each column support?
4. **Primary key:** How do we identify exactly one row?
5. **Constraints:** Which invalid records must the database reject?
6. **Evidence:** Which valid and invalid inserts will test the design?

### Justification sentence
> “I chose **[type/key/constraint]** for **[column]** because **[requirement]**. The trade-off is **[cost or limit]**.”

Example: “I chose `TEXT` for phone numbers because they are labels that may start with zero or contain `+`. The trade-off is that a text type alone does not validate phone-number format.”

A SQL statement that runs is not automatically a good design. It must match the requirements.

In [5]:
# Python demonstration: a numeric-looking identifier is not always a number.
phone = "08001234567"
print("proof that when we convert to int and back to a string, the 0 doesnt come back: ", str(int(phone)) != phone)
print(str(int(phone)))
print("As text:", phone)
print("As an integer:", int(phone))
assert phone.startswith("0")
assert str(int(phone)) != phone
print("Use numeric types for quantities; use text for labels whose formatting matters.")

proof that when we convert to int and back to a string, the 0 doesnt come back:  True
8001234567
As text: 08001234567
As an integer: 8001234567
Use numeric types for quantities; use text for labels whose formatting matters.


## 3. Choose types from meaning, not appearance

| PostgreSQL type | Suitable use here | Why / limit |
|---|---|---|
| `TEXT` | Titles, names, email, ISBN, phone | Variable-length text. Does not validate format by itself. |
| `SMALLINT` | Publication year | Whole number; range −32,768 to 32,767. Add a rule to reject non-positive years. |
| `INTEGER` | Modest counts | Whole number; roughly ±2.1 billion. |
| `BIGINT` | Generated record IDs; large minor-unit money totals | Whole number; roughly ±9.22 quintillion. More storage than INTEGER. |
| `BOOLEAN` | Whether a membership is active | True/false; add NOT NULL if a third, unknown state is not allowed. |
| `DATE` | Join date; date-only due date | Calendar date, no time of day. |
| `TIMESTAMPTZ` | When Harbour received a document | An instant, displayed in the session timezone. Not the original timezone label. |
| `NUMERIC(p, s)` | Exact decimal amounts when the unit contract requires them | Exact decimal; precision/scale must fit the domain. |

`VARCHAR(n)` is appropriate when a real maximum length is required. It is not automatically a better choice than `TEXT`.

**Money in this plan:** Store fines in integer minor units, as the README requests. For a naira-based design, explicitly call the column `amount_kobo` and agree that 100 kobo = ₦1. Never leave the unit ambiguous.

In [2]:
# Exact money: represent the unit explicitly.
from decimal import Decimal

amount_kobo = 12550
amount_naira = Decimal(amount_kobo) / Decimal(100)
print("Minor units:", amount_kobo, "kobo")
print("Display:", amount_naira, "naira")
assert amount_naira == Decimal("125.50")

print("Binary floating-point example:", 0.1 + 0.2)
print("Exact decimal example:", Decimal("0.1") + Decimal("0.2"))
# Choose INTEGER or BIGINT based on the largest possible minor-unit amount.
# This demonstration does not create a fines table; that belongs to schema work.

Minor units: 12550 kobo
Display: 125.5 naira
Binary floating-point example: 0.30000000000000004
Exact decimal example: 0.3


## 4. Missing, empty and zero are different

- `NULL`: a value is absent or unknown.
- `''`: a text value exists but contains no characters.
- `0`: a known numeric value equal to zero.
- `FALSE`: a known boolean value, not “missing”.

An unknown publication year should be `NULL`, not 0. An optional missing phone may be `NULL`. An unknown name is not allowed in our exercise.

`NOT NULL` rejects NULL but does **not** reject an empty or whitespace-only string. Use an additional `CHECK` when blank text is invalid.

**PostgreSQL detail:** A `CHECK` rejects FALSE, but accepts TRUE or unknown/NULL. That is why `CHECK (publication_year > 0)` permits a missing year; `NOT NULL` is needed separately when absence is forbidden.

**Quick check:** Does `name TEXT NOT NULL` reject `''`? **No.**

## 5. A primary key answers “Which exact row?”

A **primary key** is a column or group of columns that uniquely identifies every row.
- It is unique and cannot contain NULL.
- A table has at most one primary-key constraint; that constraint can contain multiple columns.
- Choose a key that stays stable when ordinary facts change.
- PostgreSQL creates a unique index to enforce a primary key.

### Our choice
```sql
book_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY
```
- `BIGINT`: integer ID with a large range.
- `IDENTITY`: PostgreSQL generates values when the ID is omitted.
- `PRIMARY KEY`: PostgreSQL enforces uniqueness and non-nullness.

**Identity generation alone is not a uniqueness constraint.** Keep the primary key.

Generated IDs can have gaps, including after failed inserts. Do not use “largest ID” as a row count or assume IDs prove time order. Do not renumber them to fill gaps.

## 6. Compare key choices before committing

| Candidate | Why it might work | Why we do not choose it here |
|---|---|---|
| Book title | Readable | Different records can share a title; titles can be corrected. |
| ISBN | Domain identifier for an edition/format | May be missing; book and edition scope must be agreed. Keep as an optional unique attribute in this exercise. |
| Member email | Familiar lookup | Can change; our policy treats it as a unique contact attribute, not permanent identity. |
| Generated integer | Small, simple key for a central database | Needs a generator; generated numbers reveal no business meaning. **Our choice.** |
| UUID | Useful when IDs are created across systems | Larger keys and a generation policy. Not automatically necessary. |

A **natural key** comes from the domain. A **surrogate key** is introduced to identify the record. Neither is always best.

A **composite key** uses more than one column, e.g. `(document_id, version_number)` for one document version. That is one primary key with two components—not two primary keys.

**Do not choose a primary key only because today's sample data happens to be unique.**

## 7. Library table design: books

### Restate the requirements
One row is one catalogue book record. Physical copies and loan history will have their own tables in the full schema.
- A book needs a generated ID, a title, and an author label.
- Titles and author labels must not be blank.
- Publication year is optional; if supplied, it must be positive.
- ISBN is optional; supplied ISBN values must be unique under this exercise's catalogue policy.

| Column | Choice | Justification |
|---|---|---|
| `book_id` | BIGINT identity primary key | Stable identifier even if title/ISBN changes. |
| `title` | TEXT, required, nonblank | Titles are text and not unique. |
| `author` | TEXT, required, nonblank | Simple display label for this lesson. Multiple-author modelling is outside scope. |
| `publication_year` | SMALLINT, optional, positive if supplied | A year is a whole number, not a full date. No invented January 1 date. |
| `isbn` | TEXT, optional, unique, nonblank if supplied | Identifier, not a quantity. Format/checksum validation is separate. |

**Trade-offs:** The simple author field is not a full author catalogue. The ISBN policy needs review for the final book/edition model. Future publication years are allowed here; we do not invent a “must be in the past” rule.

In [3]:
BOOKS_SQL = """
CREATE TEMP TABLE books (
    book_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    title TEXT NOT NULL CHECK (btrim(title) <> ''),
    author TEXT NOT NULL CHECK (btrim(author) <> ''),
    publication_year SMALLINT CHECK (publication_year > 0),
    isbn TEXT UNIQUE CHECK (btrim(isbn) <> '')
);
"""
print(BOOKS_SQL)
# TEMP keeps this lesson's tables session-local; it is not the production migration.
# A real migration would create persistent tables in the team's agreed schema.


CREATE TEMP TABLE books (
    book_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    title TEXT NOT NULL CHECK (btrim(title) <> ''),
    author TEXT NOT NULL CHECK (btrim(author) <> ''),
    publication_year SMALLINT CHECK (publication_year > 0),
    isbn TEXT UNIQUE CHECK (btrim(isbn) <> '')
);



## 8. Library table design: members

### Restate the requirements
One row represents one registered library member.
- Required, nonblank name and email.
- Email must be unique under an **exact, case-sensitive** comparison policy for this exercise.
- Optional phone number, kept as text.
- Join date defaults to the database session's current date if omitted.
- Active status defaults to true if omitted; NULL is not allowed.

| Column | Choice | Justification |
|---|---|---|
| `member_id` | BIGINT identity primary key | Changing contact details does not change identity. |
| `full_name` | TEXT, required, nonblank | People can share a name; do not mark it unique. |
| `email` | TEXT, required, unique, nonblank | Exercise policy: one account per exact email string. Not the primary key. |
| `phone` | TEXT, optional, nonblank if supplied | Preserves leading zeroes and `+`. Does not validate the number. |
| `joined_on` | DATE, required, default CURRENT_DATE | We need a calendar date, not an exact instant. |
| `is_active` | BOOLEAN, required, default TRUE | Two allowed states, not arbitrary labels. |

**Email limit:** This is not a complete account policy or email validator. Case variants can coexist. Agree on normalisation before production use.

In [4]:
MEMBERS_SQL = """
CREATE TEMP TABLE members (
    member_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    full_name TEXT NOT NULL CHECK (btrim(full_name) <> ''),
    email TEXT NOT NULL UNIQUE CHECK (btrim(email) <> ''),
    phone TEXT CHECK (btrim(phone) <> ''),
    joined_on DATE NOT NULL DEFAULT CURRENT_DATE,
    is_active BOOLEAN NOT NULL DEFAULT TRUE
);
"""
print(MEMBERS_SQL)
# DEFAULT supplies an omitted value. It does not replace an explicitly supplied NULL.


CREATE TEMP TABLE members (
    member_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    full_name TEXT NOT NULL CHECK (btrim(full_name) <> ''),
    email TEXT NOT NULL UNIQUE CHECK (btrim(email) <> ''),
    phone TEXT CHECK (btrim(phone) <> ''),
    joined_on DATE NOT NULL DEFAULT CURRENT_DATE,
    is_active BOOLEAN NOT NULL DEFAULT TRUE
);



## 9. Run SQL safely: optional PostgreSQL connection

The SQL strings above display without a database. The next cells can also execute them against PostgreSQL.

**To enable execution:**
1. Use a disposable learning database, not production.
2. Install the driver in your notebook's Python environment: `python -m pip install "psycopg[binary]"` in a terminal.
3. Set environment variable `HARBOUR_LESSON_DSN` before starting Jupyter, using your learning database connection string. Do not put passwords in the notebook or commit them.
4. Run from the setup cell below. Tables are temporary and disappear when the connection closes.

The helper uses parameters for data values and a transaction per statement so an expected error is rolled back without breaking later tests. It checks the PostgreSQL error code, not fragile English error wording.

**Without a connection:** SQL tests print **NOT RUN**. Python demonstrations still work. Do not report database tests as passed unless you actually ran them.

In [5]:
import os

connection = None
lesson_dsn = os.environ.get("HARBOUR_LESSON_DSN")
if lesson_dsn:
    try:
        import psycopg
    except ImportError:
        print("NOT RUN: install psycopg in this Python environment first.")
    else:
        try:
            connection = psycopg.connect(lesson_dsn, autocommit=True)
            print("Connected. The lesson creates only session-local temporary tables.")
        except psycopg.Error:
            print("NOT RUN: connection failed. Check the DSN privately; no credentials printed.")
else:
    print("NOT RUN: no HARBOUR_LESSON_DSN. SQL is shown as teaching examples only.")


def run_sql(label, statement, params=None, expected_error=None):
    if connection is None:
        print(f"NOT RUN: {label}")
        return None
    try:
        with connection.transaction():
            with connection.cursor() as cursor:
                cursor.execute(statement, params)
                rows = cursor.fetchall() if cursor.description else None
    except psycopg.Error as error:
        if expected_error is not None and error.sqlstate == expected_error:
            print(f"PASS: {label}; rejected with SQLSTATE {error.sqlstate}")
            return None
        raise
    if expected_error is not None:
        raise AssertionError(f"{label}: invalid operation was accepted")
    print(f"PASS: {label}")
    if rows is not None:
        print(rows)
    return rows

# Rerun this setup cell only after closing an existing lesson connection.
# Rerun the next cell to reset just the temporary books/members data.

NOT RUN: no HARBOUR_LESSON_DSN. SQL is shown as teaching examples only.


In [6]:
# Explicit pg_temp qualification prevents resetting any persistent tables.
run_sql("Reset temporary books", "DROP TABLE IF EXISTS pg_temp.books")
run_sql("Reset temporary members", "DROP TABLE IF EXISTS pg_temp.members")
run_sql("Create books", BOOKS_SQL)
run_sql("Create members", MEMBERS_SQL)

run_sql("Insert fictional book", """
    INSERT INTO pg_temp.books (title, author, publication_year, isbn)
    VALUES (%s, %s, %s, %s)
    RETURNING book_id, title, publication_year
""", ("The Lantern Map", "Ada Fiction", 2021, "9780000000019"))

run_sql("Insert fictional member and apply defaults", """
    INSERT INTO pg_temp.members (full_name, email, phone)
    VALUES (%s, %s, %s)
    RETURNING member_id, full_name, phone, joined_on, is_active
""", ("Boma Example", "boma@example.invalid", "08001234567"))
# ISBNs are fictional strings; this lesson does not certify checksum validity.

NOT RUN: Reset temporary books
NOT RUN: Reset temporary members
NOT RUN: Create books
NOT RUN: Create members
NOT RUN: Insert fictional book
NOT RUN: Insert fictional member and apply defaults


## 10. Read what the successful inserts demonstrate

If the previous cell ran against PostgreSQL:
- IDs were generated because they were omitted from the inserts.
- The phone remains text with its leading zero.
- `joined_on` receives CURRENT_DATE in the database session.
- `is_active` receives TRUE.

Do not assert an exact ID or date in your presentation: sequences may have gaps, and the date depends on when and where the session runs.

**Important distinctions:**
- A default is a convenient starting value, not a validation rule.
- A primary key identifies the record; an email constraint enforces a separate business policy.
- `UNIQUE` on a nullable ISBN normally permits multiple NULLs in PostgreSQL. That supports several records with unknown ISBNs.

**Predict:** Will another book with the same title but a different ISBN be accepted? **Yes.** Title is not unique.

In [7]:
run_sql("Same title, different record", """
    INSERT INTO pg_temp.books (title, author, publication_year, isbn)
    VALUES (%s, %s, %s, %s)
    RETURNING book_id, title
""", ("The Lantern Map", "Chidi Fiction", 2023, "9780000000026"))

run_sql("Missing year and ISBN are allowed", """
    INSERT INTO pg_temp.books (title, author) VALUES (%s, %s)
    RETURNING book_id, publication_year, isbn
""", ("Quiet Harbour", "Ada Fiction"))

run_sql("Another unknown ISBN is allowed", """
    INSERT INTO pg_temp.books (title, author) VALUES (%s, %s)
    RETURNING book_id, isbn
""", ("Paper Tides", "Boma Fiction"))

run_sql("Confirm member types", """
    SELECT pg_typeof(member_id)::text, pg_typeof(phone)::text,
           pg_typeof(joined_on)::text, pg_typeof(is_active)::text
    FROM pg_temp.members LIMIT 1
""")

NOT RUN: Same title, different record
NOT RUN: Missing year and ISBN are allowed
NOT RUN: Another unknown ISBN is allowed
NOT RUN: Confirm member types


## 11. Break the design on purpose

Predict the database response before running the next cell.

| Operation | Expected rejection | PostgreSQL SQLSTATE |
|---|---|---|
| NULL book title | Required value missing | 23502: not-null violation |
| Whitespace-only title | Nonblank title rule | 23514: check violation |
| Publication year 0 | Positive-year rule | 23514 |
| Duplicate supplied ISBN | Uniqueness rule | 23505: unique violation |
| Duplicate member email | Uniqueness rule | 23505 |
| Explicit NULL active status | Required boolean missing | 23502 |
| Year 40000 | Outside SMALLINT range | 22003: numeric value out of range |

A rejected bad insert is useful evidence. An error is only “expected” when it matches the specific rule being tested.

In [8]:
run_sql("Reject NULL title", """
    INSERT INTO pg_temp.books (title, author) VALUES (NULL, 'Ada Fiction')
""", expected_error="23502")
run_sql("Reject blank title", """
    INSERT INTO pg_temp.books (title, author) VALUES ('   ', 'Ada Fiction')
""", expected_error="23514")
run_sql("Reject zero year", """
    INSERT INTO pg_temp.books (title, author, publication_year)
    VALUES ('Bad Year', 'Ada Fiction', 0)
""", expected_error="23514")
run_sql("Reject duplicate ISBN", """
    INSERT INTO pg_temp.books (title, author, isbn)
    VALUES ('Other Record', 'Ada Fiction', '9780000000019')
""", expected_error="23505")
run_sql("Reject duplicate email", """
    INSERT INTO pg_temp.members (full_name, email)
    VALUES ('Another Person', 'boma@example.invalid')
""", expected_error="23505")
run_sql("Default does not repair explicit NULL", """
    INSERT INTO pg_temp.members (full_name, email, is_active)
    VALUES ('Ada Example', 'ada@example.invalid', NULL)
""", expected_error="23502")
run_sql("Reject out-of-range SMALLINT", """
    INSERT INTO pg_temp.books (title, author, publication_year)
    VALUES ('Far Future', 'Ada Fiction', 40000)
""", expected_error="22003")

NOT RUN: Reject NULL title
NOT RUN: Reject blank title
NOT RUN: Reject zero year
NOT RUN: Reject duplicate ISBN
NOT RUN: Reject duplicate email
NOT RUN: Default does not repair explicit NULL
NOT RUN: Reject out-of-range SMALLINT


In [9]:
# Test PRIMARY KEY enforcement separately from identity generation.
# OVERRIDING SYSTEM VALUE is used only to deliberately attempt bad IDs.
run_sql("Reject duplicate primary key", """
    INSERT INTO pg_temp.books (book_id, title, author)
    OVERRIDING SYSTEM VALUE
    SELECT book_id, 'Duplicate ID Attempt', 'Ada Fiction'
    FROM pg_temp.books ORDER BY book_id LIMIT 1
""", expected_error="23505")

run_sql("Reject NULL primary key", """
    INSERT INTO pg_temp.books (book_id, title, author)
    OVERRIDING SYSTEM VALUE VALUES (NULL, 'Null ID Attempt', 'Ada Fiction')
""", expected_error="23502")

run_sql("Show surviving valid rows", """
    SELECT book_id, title, publication_year, isbn
    FROM pg_temp.books ORDER BY book_id
""")
# Failed inserts may consume generated IDs. Gaps are not a broken primary key.

NOT RUN: Reject duplicate primary key
NOT RUN: Reject NULL primary key
NOT RUN: Show surviving valid rows


## 12. Transfer the reasoning to Harbour—not the library schema

The plan gives Harbour its own schema v1 and later document ingestion/Q&A capabilities. It does **not** specify an approved document schema. The following is a **teaching proposal for team review**, not an extracted requirement.

**Assumed row meaning:** One row is one ingested text document. This simplified proposal does not yet model versions, chunks, embeddings, permissions, tenants, or deduplication.

| Column | Proposed type/rule | Reason |
|---|---|---|
| `document_id` | BIGINT identity primary key | Stable internal record identity. UUID is an alternative if distributed ID generation is needed. |
| `title` | TEXT, required, nonblank | Display label; different documents may share a title. |
| `body` | TEXT, required, nonblank | Document text for later processing. Real size limits still need a policy. |
| `source_url` | TEXT, optional | Some documents may be uploaded rather than fetched. Text alone does not validate a URL. |
| `received_at` | TIMESTAMPTZ, required, default CURRENT_TIMESTAMP | Track an actual ingestion instant across timezones. |

**Why not source URL as the primary key?** A document may have no URL, and the same URL can later contain changed content.

**Team decision before implementation:** Is a row a source, a document, a version, or an ingestion attempt? The answer changes the table design.

In [10]:
HARBOUR_DOCUMENTS_SQL = """
CREATE TEMP TABLE harbour_documents (
    document_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    title TEXT NOT NULL CHECK (btrim(title) <> ''),
    body TEXT NOT NULL CHECK (btrim(body) <> ''),
    source_url TEXT CHECK (btrim(source_url) <> ''),
    received_at TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP
);
"""
print(HARBOUR_DOCUMENTS_SQL)
run_sql("Reset temporary Harbour proposal", "DROP TABLE IF EXISTS pg_temp.harbour_documents")
run_sql("Create Harbour teaching proposal", HARBOUR_DOCUMENTS_SQL)
run_sql("Insert fictional Harbour document", """
    INSERT INTO pg_temp.harbour_documents (title, body)
    VALUES (%s, %s)
    RETURNING document_id, title, source_url, received_at
""", ("Fictional Returns Policy", "Example customers may request a return within seven days."))
run_sql("Reject empty document body", """
    INSERT INTO pg_temp.harbour_documents (title, body) VALUES ('Empty Policy', '')
""", expected_error="23514")


CREATE TEMP TABLE harbour_documents (
    document_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    title TEXT NOT NULL CHECK (btrim(title) <> ''),
    body TEXT NOT NULL CHECK (btrim(body) <> ''),
    source_url TEXT CHECK (btrim(source_url) <> ''),
    received_at TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP
);

NOT RUN: Reset temporary Harbour proposal
NOT RUN: Create Harbour teaching proposal
NOT RUN: Insert fictional Harbour document
NOT RUN: Reject empty document body


## 13. Explain dates and instants correctly

- Library `joined_on DATE`: “Which calendar day did the person join?”
- Harbour `received_at TIMESTAMPTZ`: “At what instant did ingestion happen?”
- `TIMESTAMP WITHOUT TIME ZONE`: a local date/time without an associated timezone. Do not choose it accidentally when you mean an instant.

PostgreSQL `TIMESTAMPTZ` preserves the instant, not the originally supplied zone name. It displays according to the session timezone. Store a separate zone identifier if that original information is a requirement.

**Example:** `09:00 UTC` and `10:00 WAT` describe the same instant. A DATE cannot distinguish those times; a TEXT value cannot enforce timestamp semantics.

**Check:** If a loan is due by a specific time rather than just a day, revisit the due-date type and timezone rules. Choose from the requirement, not a habit.

## 14. Team exercise: repair a weak design

```sql
CREATE TABLE weak_members (
    full_name TEXT PRIMARY KEY,
    email TEXT,
    phone BIGINT,
    joined_on TEXT,
    is_active TEXT
);
```

Requirements: names may repeat and change; email is required and unique under the agreed comparison policy; phone may start with zero; join date is a calendar date; status is true/false and required.

**Tasks:**
1. State the row meaning.
2. Identify at least five problems or missing rules.
3. Write a corrected CREATE TABLE statement in the next workspace.
4. Justify the primary key and each type in one sentence.
5. Give one valid insert and two rejected inserts, with reasons.

**Extension:** Another team proposes using email as the primary key. Explain one advantage and one disadvantage instead of saying it is always wrong.

In [11]:
# Learner workspace: replace this string with your own SQL before checking answers.
my_members_sql = """
-- Write CREATE TABLE here. Use TEMP if executing in the lesson connection.
"""

# Complete before running tests:
# Row meaning:
# Primary-key choice and trade-off:
# Column/type justifications:
# Valid insert and expected result:
# Rejected insert 1 and violated rule:
# Rejected insert 2 and violated rule:

print(my_members_sql)
# Do not execute a blank/comment-only statement through run_sql.
# To execute a completed design, use a new temporary table name, e.g. my_members.


-- Write CREATE TABLE here. Use TEMP if executing in the lesson connection.



## 15. Answer check and Harbour design challenge

### Weak-members repair: check after attempting
A suitable correction is the `members` design in Section 8:
- Replace the changeable/non-unique name key with a generated ID primary key.
- Require a nonblank name without marking it unique.
- Require a nonblank, unique email under an explicit comparison policy.
- Change phone to TEXT to preserve formatting.
- Change join date to DATE.
- Change active status to BOOLEAN NOT NULL.

**Trade-off:** An email primary key avoids a separate identifier, but changing email changes the identifier used elsewhere. A surrogate ID separates identity from contact details.

### Independent Harbour task
Design a **proposed** `ingestion_jobs` table. Assume one row represents one attempt to ingest a document.
- A generated stable ID identifies the attempt.
- A source label is required and nonblank.
- Status must be one of `queued`, `running`, `succeeded`, `failed`.
- Creation time is a required instant; completion time is absent until finished.

Write a column/type/key table first, then SQL. Add a CHECK for the allowed status values. Explain why this job is not the same thing as a document.

This is a design exercise, not an approved addition to Harbour. Foreign keys, full lifecycle consistency and migrations are for team review with the other schema owners.

## 16. Twenty questions for your question round

Ask for a reason, not only a keyword. Answer cues are in parentheses; hide them while presenting.

1. What does one row in our books table represent? *(One catalogue record, not a physical copy.)*
2. Why must we define row meaning before columns? *(It decides which facts belong together.)*
3. Why is phone TEXT rather than BIGINT? *(Preserve labels, leading zeroes and +.)*
4. Why is publication_year not DATE here? *(We know a year, not a complete date.)*
5. What does NULL mean for that year? *(Unknown/absent; not zero.)*
6. Does NOT NULL reject an empty title? *(No; add a nonblank CHECK.)*
7. Why can two books have the same title? *(Title is a label, not unique identity.)*
8. What two rules does PRIMARY KEY enforce? *(Unique and non-null.)*
9. Can one primary key use two columns? *(Yes; a composite key.)*
10. Why use a generated ID rather than a member name? *(Names repeat/change.)*
11. Does identity generation alone enforce uniqueness? *(No; keep the primary key.)*
12. Do ID gaps mean rows are missing incorrectly? *(No; gaps can result from failed inserts.)*
13. Why not use the largest ID to count rows? *(Gaps and deletions; use COUNT.)*
14. What is one trade-off of UUID versus BIGINT? *(Distributed generation versus larger keys/extra policy.)*
15. Why can multiple unknown ISBNs coexist? *(Nullable UNIQUE normally allows multiple NULLs.)*
16. Does DEFAULT TRUE replace explicit NULL? *(No; NOT NULL rejects it.)*
17. When choose DATE rather than TIMESTAMPTZ? *(Calendar day versus instant.)*
18. What does TIMESTAMPTZ not preserve? *(Original timezone name.)*
19. Why store fines in named integer minor units? *(Exact arithmetic and explicit units; range/conversion must be managed.)*
20. Why is the Harbour document table only a proposal? *(The plan names capabilities, not an approved schema; row meaning and policies need review.)*

## 17. Exit ticket and Friday review checklist

**Without copying:** Design a books table with an ID, title, author, optional publication year, and optional ISBN. Then justify every type and the primary key.

A teammate should be able to check:
- [ ] One sentence defines the row meaning.
- [ ] The primary key is stable, unique and non-null.
- [ ] Types match operations, ranges and formatting requirements.
- [ ] NULL, blank, zero and defaults are not confused.
- [ ] Required/unique/check rules match stated requirements, not guesses.
- [ ] At least one valid insert and two invalid inserts are specified.
- [ ] Test status is honest: executed results or NOT RUN.
- [ ] One design trade-off and one unresolved policy are recorded.

**Friday handoff:** Review your `books`/`members` definitions with the library schema owner (Member 6) and constraints/normalisation owner (Member 8). Integrate the agreed definitions into the migration process; do not create competing production tables from this notebook.

For Harbour, repeat the design reasoning in its own repository. Do not import library-domain tables just because the SQL works.

**AI disclosure:** This notebook was AI-assisted. The owner should hand-write the core table definitions, explain them without reading the screen, run the failure experiments, and record what was checked before merge.

**Sources:** [Harbour Week 3 plan](https://github.com/Snrboi/harbour/blob/learn/README.md#week-3-sql-and-postgresql-basics); PostgreSQL documentation on [data types](https://www.postgresql.org/docs/current/datatype.html), [constraints](https://www.postgresql.org/docs/current/ddl-constraints.html), and [identity columns](https://www.postgresql.org/docs/current/ddl-identity-columns.html).

In [12]:
# Close the lesson connection when finished. Temporary tables disappear.
if connection is not None:
    connection.close()
    connection = None
    print("Closed. Session-local lesson tables have been removed.")
else:
    print("No active PostgreSQL connection. Database examples were NOT RUN unless enabled earlier.")

print("Final takeaway: define the row, choose types from meaning, choose a stable key, then test the rules.")

No active PostgreSQL connection. Database examples were NOT RUN unless enabled earlier.
Final takeaway: define the row, choose types from meaning, choose a stable key, then test the rules.
